# Scam Conversation Detector (Whisper + Intent Progression)
This notebook detects scam conversation progression using sentence embeddings.

Stages:
GREETING → AUTHORITY → PROBLEM → URGENCY → DATA_REQUEST

In [27]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [28]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6489.12it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [29]:
INTENTS = {

"greeting": [
"hello sir how are you",
"hello maam how are you",
"good morning sir",
"good afternoon maam",
"good evening sir",
"am I speaking with the account holder",
"is this the owner of this number",
"am I speaking with Arya",
"can you hear me clearly",
"this is a service call regarding your account",
"this call is for account verification",
"namaste sir",
"namaste maam",
"hello this is customer service",
"hello I hope you are doing well",
"hello I am calling regarding your bank account",
"hello this call is about your bank services",
"hello we are calling from bank support",
"hello this is an official call",
"hello this is customer care calling",

],

"authority": [
"I am calling from SBI bank",
"I am calling from HDFC bank",
"I am calling from ICICI bank",
"I am calling from Axis bank",
"I am calling from bank verification team",
"I am calling from card security department",
"I am calling from ATM department",
"I am calling from KYC verification department",
"I am calling from bank security team",
"I am calling from banking support",
"This is the bank verification team",
"This is your bank customer care",
"This is bank technical department",
"This is UPI support team",
"I am calling from payment verification team",
"I am calling from banking headquarters",
"This call is from official bank department",
"This is your bank security desk",
"This is financial services department",
"This is your registered bank support",
"main bank se bol raha hoon",
"main SBI bank se bol raha hoon",
"main bank verification department se bol raha hoon",
"main customer support se bol raha hoon",
"main bank security team se bol raha hoon",
"ye bank verification call hai",
"main card department se bol raha hoon"
],

"problem": [
"your account has suspicious activity",
"we detected unusual transactions",
"your ATM card has been temporarily blocked",
"your bank account is under review",
"your card has been restricted",
"someone attempted login to your account",
"your account has security issues",
"your account verification failed",
"your KYC is incomplete",
"your KYC update is pending",
"your account is temporarily suspended",
"your debit card is flagged",
"your account has been locked",
"your account needs re verification",
"your bank services may stop",
"your account security is compromised",
"someone tried accessing your banking account",
"your account has abnormal activity",
"your account is under investigation",
"your card access is limited",
"aapka account block ho sakta hai",
"aapka ATM card block ho gaya hai",
"aapka KYC pending hai",
"aapke account mein suspicious activity hai",
"aapka bank account verify nahi hua",
"aapka account temporarily block ho gaya hai",
"aapka account security issue mein hai"
],

"urgency": [
"you must verify immediately",
"this is very urgent",
"immediate action is required",
"please verify now",
"your account will be blocked today",
"your account will be permanently blocked",
"this needs to be done right now",
"you need to confirm immediately",
"please do this quickly",
"we need to complete verification now",
"your banking service will stop",
"your card will be blocked today",
"your account will be frozen",
"this is time sensitive",
"verification must be done now",
"please respond quickly",
"this process must be completed immediately",
"this cannot be delayed",
"we must finish verification now",
"your service may stop today",
"abhi verify karna hoga",
"turant verify karna padega",
"abhi confirm karna hoga",
"warna account block ho jayega",
"jaldi karna hoga",
"abhi process complete karna hoga"
],

"data_request": [
"tell me the OTP sent to your phone",
"share the OTP you received",
"read the verification code",
"tell me the six digit code",
"read the SMS you received",
"confirm the number sent to your phone",
"share the verification code with me",
"tell me the security code",
"read the digits sent to your mobile",
"confirm the code sent to your number",
"tell me the message you received",
"share the OTP quickly",
"tell me the code for verification",
"confirm the SMS code",
"read the message from the bank",
"tell me the numbers you received",
"share the code sent to your phone",
"confirm the OTP message",
"tell me the verification number",
"read the code you just received",
"OTP bata dijiye",
"OTP bol dijiye",
"jo OTP aya hai wo bataiye",
"SMS mein jo code aya hai wo boliye",
"verification code bata dijiye",
"mobile pe jo code aya hai wo boliye",
"jo number aya hai wo bataiye",
"OTP share kar dijiye",
"message mein jo code hai wo boliye",
"OTP confirm kar dijiye"
]

}

In [30]:
intent_embeddings = {}

for intent, sentences in INTENTS.items():
    intent_embeddings[intent] = model.encode(sentences)

print('Intent embeddings created')

Intent embeddings created


In [31]:
def detect_intents(text):
    text = text.lower().strip()
    text_embedding = model.encode([text])

    scores = {}

    for intent, embeddings in intent_embeddings.items():
        similarity = cosine_similarity(text_embedding, embeddings)
        scores[intent] = float(np.max(similarity))

    return scores

In [32]:
STAGE_MAP = {
 "greeting":1,
 "authority":2,
 "problem":3,
 "urgency":4,
 "data_request":5
}

In [33]:
def detect_stage(intent_scores):

    best_intent = max(intent_scores, key=intent_scores.get)
    best_score = intent_scores[best_intent]

    if best_score < 0.55:
        return 0, best_intent, best_score

    return STAGE_MAP[best_intent], best_intent, best_score

In [34]:
current_stage = 0
stage_progress = []

In [35]:
def update_stage(stage):

    global current_stage

    if stage == current_stage + 1:
        current_stage = stage
        stage_progress.append(stage)

    elif stage > current_stage:
        current_stage = stage
        stage_progress.append(stage)

In [36]:
def stage_risk():

    if current_stage == 0:
        return 0.05

    if current_stage == 1:
        return 0.1

    if current_stage == 2:
        return 0.25

    if current_stage == 3:
        return 0.5

    if current_stage == 4:
        return 0.7

    if current_stage == 5:
        return 0.95

In [37]:
def process_text(text):

    intents = detect_intents(text)

    stage, intent, score = detect_stage(intents)

    update_stage(stage)

    risk = stage_risk()

    print('Transcript:', text)
    print('Detected intent:', intent)
    print('Stage:', stage)
    print('Scam Risk:', risk)

    if risk > 0.8:
        print('⚠ SCAM ALERT')

    print('-'*40)

In [38]:
process_text('hello beta')
process_text('I am your dad')
process_text('tum kaise ho')
process_text('ache se rho')
process_text('i love you')
process_text('OTP dena')

Transcript: hello beta
Detected intent: greeting
Stage: 0
Scam Risk: 0.05
----------------------------------------
Transcript: I am your dad
Detected intent: urgency
Stage: 0
Scam Risk: 0.05
----------------------------------------
Transcript: tum kaise ho
Detected intent: data_request
Stage: 0
Scam Risk: 0.05
----------------------------------------
Transcript: ache se rho
Detected intent: authority
Stage: 0
Scam Risk: 0.05
----------------------------------------
Transcript: i love you
Detected intent: greeting
Stage: 0
Scam Risk: 0.05
----------------------------------------
Transcript: OTP dena
Detected intent: data_request
Stage: 5
Scam Risk: 0.95
⚠ SCAM ALERT
----------------------------------------


In [39]:
43

43